# scvi/00 — RDS → h5ad Conversion

Converts `reference.rds` and `query.rds` (from main R pipeline NB00)
into AnnData `.h5ad` format required by scvi-tools.

**Strategy:** shell out to `Rscript` to export raw counts as Matrix Market +
metadata CSV, then reconstruct `AnnData` in Python — no rpy2 needed.

**Inputs:** `data/reference.rds`, `data/query.rds`  
**Outputs:** `data/scvi/reference.h5ad`, `data/scvi/query.h5ad`

In [ ]:
# Parameters — injected by papermill
reference_rds     = "data/cca/reference.rds"
query_rds         = "data/cca/query.rds"
output_ref_h5ad   = "data/scvi/reference.h5ad"
output_query_h5ad = "data/scvi/query.h5ad"
rscript_path      = "Rscript"  # override if needed


## 0 · Install & Import

In [ ]:
import subprocess, sys, importlib.util, time, os

# Install scvi-tools + scanpy if missing
for pkg, pip_name in [("anndata","anndata"),("scanpy","scanpy"),("scvi","scvi-tools")]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable,"-m","pip","install",pip_name,"--quiet"])

import anndata as ad
import scanpy as sc
import scvi
import scipy.io, scipy.sparse
import pandas as pd
import numpy as np

print(f"anndata {ad.__version__} | scanpy {sc.__version__} | scvi-tools {scvi.__version__}")


## 1 · Export RDS → Matrix Market via Rscript

In [ ]:
R_EXPORT_SCRIPT = """
# Load project renv library so Seurat is found
local({
  renv_lib <- file.path(getwd(), "renv", "library", "macos", "R-4.4", "aarch64-apple-darwin20.0.0")
  if (dir.exists(renv_lib)) .libPaths(c(renv_lib, .libPaths()))
})
suppressPackageStartupMessages({ library(Seurat); library(Matrix) })
args     <- commandArgs(trailingOnly=TRUE)
rds_path <- args[1]; out_dir <- args[2]
dir.create(out_dir, recursive=TRUE, showWarnings=FALSE)
cat(sprintf(\"Loading %s ...\\n\", rds_path))
obj <- readRDS(rds_path)
# Seurat 5 uses layers; Seurat 4 uses slots
counts <- tryCatch(
  GetAssayData(obj, assay=\"RNA\", layer=\"counts\"),
  error = function(e) GetAssayData(obj, assay=\"RNA\", slot=\"counts\")
)
writeMM(counts, file.path(out_dir, \"matrix.mtx\"))
writeLines(rownames(counts), file.path(out_dir, \"features.tsv\"))
writeLines(colnames(counts), file.path(out_dir, \"barcodes.tsv\"))
write.csv(as.data.frame(obj@meta.data), file.path(out_dir, \"metadata.csv\"), quote=FALSE)
cat(sprintf(\"Exported: %d genes x %d cells -> %s\\n\",
            nrow(counts), ncol(counts), out_dir))
cat(sprintf(\"Metadata cols: %s\\n\", paste(colnames(obj@meta.data), collapse=\", \")))
"""

def export_rds_to_mex(rds_path, out_dir, rscript="Rscript"):
    """Export Seurat RDS to Matrix Market + metadata CSV via Rscript."""
    os.makedirs(out_dir, exist_ok=True)
    script_file = os.path.join(out_dir, "_export.R")
    with open(script_file, "w") as fh:
        fh.write(R_EXPORT_SCRIPT)
    result = subprocess.run(
        [rscript, "--vanilla", script_file, rds_path, out_dir],
        capture_output=True, text=True,
        cwd="/Users/yunhuazhu/Documents/gitrepos/scRNA-seq_reference-projection"
    )
    print(result.stdout.strip())
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"R export failed for {rds_path}")


In [ ]:
t0 = time.time()
print(f"Exporting reference: {reference_rds}")
export_rds_to_mex(reference_rds, "data/scvi/_ref_mex", rscript_path)
print(f"  done in {time.time()-t0:.1f}s")


In [ ]:
t0 = time.time()
print(f"Exporting query: {query_rds}")
export_rds_to_mex(query_rds, "data/scvi/_query_mex", rscript_path)
print(f"  done in {time.time()-t0:.1f}s")


## 2 · Build AnnData

In [ ]:
def mex_to_adata(mex_dir):
    """Read Matrix Market export (genes x cells), return AnnData (cells x genes)."""
    mat      = scipy.io.mmread(os.path.join(mex_dir, "matrix.mtx")).T.tocsr()
    barcodes = pd.read_csv(os.path.join(mex_dir, "barcodes.tsv"), header=None)[0]
    features = pd.read_csv(os.path.join(mex_dir, "features.tsv"),  header=None)[0]
    metadata = pd.read_csv(os.path.join(mex_dir, "metadata.csv"),  index_col=0)
    adata = ad.AnnData(
        X   = scipy.sparse.csr_matrix(mat),
        obs = metadata,
        var = pd.DataFrame(index=features.values)
    )
    adata.obs_names = barcodes.values
    adata.var_names = features.values
    adata.layers["counts"] = adata.X.copy()  # raw integer counts for scVI
    return adata

adata_ref   = mex_to_adata("data/scvi/_ref_mex")
adata_query = mex_to_adata("data/scvi/_query_mex")
print(f"Reference : {adata_ref.n_obs:,} cells x {adata_ref.n_vars:,} genes")
print(f"Query     : {adata_query.n_obs:,} cells x {adata_query.n_vars:,} genes")
print(f"Ref obs columns: {list(adata_ref.obs.columns)}")


## 3 · Save h5ad

In [ ]:
os.makedirs(os.path.dirname(output_ref_h5ad), exist_ok=True)
adata_ref.write_h5ad(output_ref_h5ad)
adata_query.write_h5ad(output_query_h5ad)
print(f"Saved: {output_ref_h5ad}  ({adata_ref.n_obs:,} cells)")
print(f"Saved: {output_query_h5ad}  ({adata_query.n_obs:,} cells)")
